# Phase 29 — Ensemble: Hierarchical BERT + Best Model

**Μοντέλα:**
- Best model: Hazard=BERT-base Focal, Product=Ensemble(BERT 0.5 + Multi-task 0.5) → 0.81882
- Hierarchical BERT → 0.79288

**Λογική:** Το hierarchical εκμεταλλεύεται τη σχέση hazard→product.
Το ensemble των δύο μπορεί να βελτιώσει κυρίως το product prediction.

**Στρατηγική:** Κρατάμε το BERT-base Focal για hazard (το καλύτερο)
και κάνουμε ensemble μόνο για το product.

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

train = pd.read_csv('train.csv')
valid = pd.read_csv('valid.csv')
test  = pd.read_csv('test.csv')

train_full = pd.concat([train, valid], ignore_index=True)
le_hazard  = LabelEncoder()
le_product = LabelEncoder()
le_hazard.fit(train_full['hazard-category'])
le_product.fit(train_full['product-category'])

# Hazard — BERT-base Focal (best)
bert_hazard_probs = np.load('npy/bertbase_focal_test_hazard_probs2.npy')

# Product — Best model: BERT(0.5) + Multi-task(0.5)
bert_product_probs  = np.load('npy/bertbase_focal_test_product_probs2.npy')
multi_product_probs = np.load('npy/multitask_test_product_probs.npy')
best_product_probs  = 0.5 * bert_product_probs + 0.5 * multi_product_probs

# Product — Hierarchical BERT
hier_product_probs = np.load('npy/hierarchical_test_product_probs.npy')

print(f'BERT hazard probs:      {bert_hazard_probs.shape}')
print(f'Best product probs:     {best_product_probs.shape}')
print(f'Hierarchical product:   {hier_product_probs.shape}')

BERT hazard probs:      (997, 10)
Best product probs:     (997, 22)
Hierarchical product:   (997, 22)


In [2]:
# Hazard predictions — BERT-base Focal (σταθερό)
test_hazard = le_hazard.inverse_transform(bert_hazard_probs.argmax(axis=1))

# Δοκιμή διαφορετικών βαρών για product
weight_combinations = [
    (1.0, 0.0),   # Best model μόνο (baseline 0.81882)
    (0.9, 0.1),
    (0.8, 0.2),
    (0.7, 0.3),
    (0.6, 0.4),
    (0.5, 0.5),
    (0.4, 0.6),
    (0.0, 1.0),   # Hierarchical μόνο
]


for w_best, w_hier in weight_combinations:
    product_avg  = w_best * best_product_probs + w_hier * hier_product_probs
    test_product = le_product.inverse_transform(product_avg.argmax(axis=1))

    filename = f'submission_hier_ensemble_b{int(w_best*10)}_h{int(w_hier*10)}.csv'
    pd.DataFrame({
        'id': test['id'],
        'hazard-category': test_hazard,
        'product-category': test_product
    }).to_csv(filename, index=False)
    print(f'Best={w_best:.1f}, Hier={w_hier:.1f} → {filename}')

print('1. submission_hier_ensemble_b9_h1.csv  (Best=0.9, Hier=0.1)')
print('2. submission_hier_ensemble_b8_h2.csv  (Best=0.8, Hier=0.2)')
print('3. submission_hier_ensemble_b7_h3.csv  (Best=0.7, Hier=0.3)')

Best=1.0, Hier=0.0 → submission_hier_ensemble_b10_h0.csv
Best=0.9, Hier=0.1 → submission_hier_ensemble_b9_h1.csv
Best=0.8, Hier=0.2 → submission_hier_ensemble_b8_h2.csv
Best=0.7, Hier=0.3 → submission_hier_ensemble_b7_h3.csv
Best=0.6, Hier=0.4 → submission_hier_ensemble_b6_h4.csv
Best=0.5, Hier=0.5 → submission_hier_ensemble_b5_h5.csv
Best=0.4, Hier=0.6 → submission_hier_ensemble_b4_h6.csv
Best=0.0, Hier=1.0 → submission_hier_ensemble_b0_h10.csv
1. submission_hier_ensemble_b9_h1.csv  (Best=0.9, Hier=0.1)
2. submission_hier_ensemble_b8_h2.csv  (Best=0.8, Hier=0.2)
3. submission_hier_ensemble_b7_h3.csv  (Best=0.7, Hier=0.3)
